# 基于 MongoDB Atlas 与 Claude Managed Agents 的欺诈审核智能体

本 cookbook 演示如何把 **MongoDB Atlas** 接入运行在 [Claude Managed Agents](README.md)（CMA）上的智能体——同时作为它的检索引擎、图存储和记录系统——且只使用标准的 CMA 模式（自定义工具和 MCP 工具集），**不需要任何平台级的 MongoDB 集成**。可运行的辅助代码位于本笔记本旁的 [﷐0﷑](mongodb_on_cma/) 目录中。每一节都链接到完整文件，因此正文始终聚焦于 *MongoDB 在哪些位置接入* 以及 *智能体在哪些位置创造价值*。

**MongoDB 在哪些位置接入：** 多数智能体技术栈会把三四个系统拼在一起：用于语义检索的向量数据库、用于关键词的独立引擎、用于关系的图存储，以及存放记录本身的操作型数据库。每一处接缝都意味着一次额外集成、一份额外凭据，以及又一个智能体对世界的认知可能发生偏差的地方。MongoDB 把这些收敛为一个引擎：同一批文档既可以按**向量**检索（`$vectorSearch`），按**全文**检索（`$search`），按两者的**混合**检索并用倒数排名融合（RRF）合并（`$rankFusion`），还可以作为**图**进行遍历（`$graphLookup`）——而且它们正是智能体读取、写入并持久化其决策的那些文档。一种查询语言，一个集群，一条连接（`pymongo`）。

**智能体在哪些位置创造价值：** CMA 在 Anthropic 一侧运行智能体循环和沙箱。你的应用掌握通往 MongoDB 的**数据通路**。智能体对 MongoDB 检索到的内容进行推理，通过 CMA 原生的 `requires_action` 门控在高风险调用上暂停等待人工，并把结论写回——整个过程中数据库凭据从不进入智能体上下文或其沙箱。

> 这里的 MongoDB 是你的**操作型数据存储和检索引擎**——是你的记录系统，
> 由你的应用掌控。这与 CMA 在平台上原生管理的记忆功能是两回事，
> 两者互为补充。

**学完本指南后你将能够：**

- 以三种方式把凭据安全的 MongoDB Atlas 数据通路接入 Claude Managed Agent。
- 把向量、全文、混合和图检索提升为智能体的自定义工具。
- 用 CMA 原生的人工介入暂停机制为高风险智能体决策设置门控。
- 让一个 MongoDB Atlas 集群成为智能体的记录系统与审计主干。

完整示例是一个**人工介入的欺诈审核智能体**，但这些模式与行业无关：换掉集合和工具，同样的形态就能服务于客服、研究或运维智能体。

## 本指南涵盖的内容

1. [把 MongoDB 连接到托管智能体](#1-connect-mongodb-to-a-managed-agent)
2. [检索：四种模式，一个引擎](#2-retrieve-four-patterns-one-engine)
3. [端到端智能体：人工介入的欺诈审核](#3-the-end-to-end-agent-human-in-the-loop-fraud-review)
4. [MongoDB Atlas 作为记录系统与审计主干](#4-mongodb-atlas-as-the-system-of-record-and-audit-backbone)
5. [小结](#5-recap)

In [1]:
%%capture
# Third-party dependencies: the Anthropic SDK (drives Claude Managed Agents via client.beta.*),
# pymongo (the host-side data path), python-dotenv, and cryptography + pyjwt (sign/verify the
# AP2 mandates). This installs the PyPI packages only — the notebook also imports the adjacent
# `mongodb_on_cma/` package and `utilities.py`, so run it from a clone of this repo (see Setup).
%pip install -q "anthropic>=0.109.0" pymongo python-dotenv cryptography pyjwt

## 前置条件

**需要掌握的知识：** Python 和 `pymongo` 基础，以及对 Claude API 的基本了解。刚接触 MongoDB + Claude？[library-RAG 笔记本](../third_party/MongoDB/rag_using_mongodb.ipynb) 会先讲解 `$vectorSearch`。刚接触自定义工具门控？[﷐2﷑](CMA_gate_human_in_the_loop.ipynb) 会讲清整个往返流程。

**需要的工具：** Python 3.11+、一个 [Anthropic API 密钥](https://console.anthropic.com)，以及一个 MongoDB Atlas 集群。

## 环境准备

**必需：** `MONGO_URI`（一个 Atlas SRV 连接字符串）以及 Anthropic API 访问权限。**一个免费的 M0 集群就能跑通本 cookbook 的全部内容**——向量、全文、混合 `$rankFusion` 以及图遍历——因此没有付费层级要求。混合检索使用原生 `$rankFusion` 阶段，它需要 MongoDB 8.0+；目前所有 Atlas 集群（包括 M0）都在 8.0 或更高版本，因此默认满足。

```bash
uv sync --all-extras   # from the repo root
cp .env.example .env    # then add MONGO_URI (and Anthropic auth, if not already configured)
```

**可选的嵌入向量/重排序服务：** 种子数据自带预先计算的嵌入向量，因此本 cookbook 无需该服务也能运行。要启用实时嵌入向量和重排序器路径，请设置 **其中一个**：`MDB_ATLAS_API_KEY`（MongoDB Atlas AI 端点，同时提供 `/v1/embeddings` 和 `/v1/rerank`）或 `VOYAGE_API_KEY`（`voyageai` SDK）。`ENABLE_RERANK=1` 会加入重排序器第二级。`AUTO_APPROVE=1` 在 CI 中确定性地解除人工门控。`COOKBOOK_MODEL` 可覆盖智能体模型（默认 `claude-haiku-4-5`）。

**还没有 Atlas 集群？** 配套笔记本 [﷐0﷑](../third_party/MongoDB/rag_using_mongodb.ipynb) 会讲解如何创建免费集群并获取你的 `MONGO_URI`；[Atlas Search 索引文档](https://www.mongodb.com/docs/atlas/atlas-search/create-index/) 介绍索引的创建。

教学代码**内联在本笔记本中**：四个检索工作流构建器（第 2 节），以及自定义工具处理函数加 `requires_action` 门控循环（第 3 节）。笔记本旁的 [﷐1﷑](mongodb_on_cma/) 包只存放你需要导入而非阅读的样板代码：[﷐2﷑](mongodb_on_cma/config.py)（索引名称与可调参数）、[﷐3﷑](mongodb_on_cma/embeddings.py)（Atlas/Voyage 客户端）、[﷐4﷑](mongodb_on_cma/tools.py)（Atlas 种子数据/索引设置 + 共用的决策与审计文档构造函数）、[﷐5﷑](mongodb_on_cma/ap2_mandates.py)（AP2 签名/验证——一个通过 `verify_mandates` 调用的加密黑盒），以及 [﷐7﷑](mongodb_on_cma/seed.py)（示例数据加载器）。导入它们、连接客户端，并解析运行配置。

In [2]:
import hashlib
import json
import logging
import os
from datetime import UTC, datetime
from typing import Any

import dotenv
from anthropic import Anthropic
from mongodb_on_cma import (
    EMBED_DIM,
    build_audit_event,
    build_decision_doc,
    ensure_indexes,
    has_anthropic_auth,
    load_seed,
    make_embedding_client,
    missing_required_env,
    preflight,
    prepare_seed,
    rerank,
    resolve_model,
    seed_collection,
    server_version,
    supports_rank_fusion,
)
from mongodb_on_cma.ap2_mandates import (
    attach_mandates,
    store_mandate_receipt,
    tool_verify_mandates,
)
from pymongo import MongoClient
from utilities import wait_for_idle_status

dotenv.load_dotenv()

missing = missing_required_env()
assert not missing, f"Set these in your environment / .env: {missing}"
# Anthropic auth is a soft check: an API key, an auth token, or a profile (e.g. the `ant`
# CLI's workload-identity federation) all work, so warn rather than block if none is set.
if not has_anthropic_auth():
    print(
        "warning: no ANTHROPIC_API_KEY / ANTHROPIC_AUTH_TOKEN / ANTHROPIC_PROFILE found; "
        "relying on the SDK / `ant` CLI to resolve credentials."
    )

MODEL = resolve_model()
ENABLE_RERANK = os.getenv("ENABLE_RERANK", "").lower() in ("1", "true")
AUTO_APPROVE = os.getenv("AUTO_APPROVE", "").lower() in ("1", "true")

# Quiet the SDK's one-shot notice that ANTHROPIC_API_KEY shadows profile/federation
# auto-discovery — expected when an API key is set; harmless under `ant` / WIF auth.
logging.getLogger("anthropic.lib.credentials._auth").setLevel(logging.ERROR)
client = Anthropic()
mongo = MongoClient(os.environ["MONGO_URI"])
db = mongo["fraud_review_demo"]
coll = db["transactions"]
ai_client = make_embedding_client()  # Atlas / Voyage / None (seed ships precomputed vectors)

print(
    f"model={MODEL}  rerank={'on' if ENABLE_RERANK else 'off'}  provider={'yes' if ai_client else 'none'}"
)

model=claude-haiku-4-5  rerank=on  provider=yes


## 1. 把 MongoDB 连接到托管智能体

智能体循环及其沙箱都**托管在 Anthropic 侧**——这正是“托管”的含义。你只需托管通往 MongoDB 的**数据通路**。在每条路径上，MongoDB 凭据都留在边界的*你这一侧*：它从不进入智能体上下文、云沙箱的环境变量，或智能体可读的任何文件。

> **本 cookbook 为什么通过 `client.beta.*` 与 CMA 通信，而不用 Claude Agent SDK。**
> Managed Agents（CMA）是一个[托管 REST API](https://platform.claude.com/docs/en/managed-agents/overview)：
> Anthropic 运行智能体循环和沙箱，你通过创建智能体/会话，并在标准 Anthropic SDK 的 beta 接口上
> 流式收发事件来驱动它
>（`client.beta.agents`、`client.beta.sessions`、`client.beta.sessions.events`）。并不存在
> 独立的“CMA SDK”——这*就是*使用 CMA 的方式。
> [Claude Agent SDK](https://code.claude.com/docs/en/agent-sdk/overview)
>（`claude-agent-sdk`，在仓库的
> [﷐0﷑](../claude_agent_sdk/) cookbooks 中有展示）是另一个产品：一个
> 在**你自己的进程**内、在你自己的基础设施上运行智能体循环的库。两者
> 互为补充——你会用 Agent SDK 在本地做原型，在生产上用 CMA 运行——但
> 像本例这样的 CMA 集成是基于 beta sessions/events API 构建的。这一选择也正是
> 下文凭据边界得以成立的原因：因为每次自定义工具调用都由*你的*进程处理，
> 所以 MongoDB 密钥始终留在宿主侧（见路径 A）。

搭建数据通路有三种方式。**本 cookbook 全程使用路径 A**——它是推荐的最轻量模式，也是把凭据完全留在宿主侧的那一种。路径 B 和 C 在其后做了简要说明以求完整。

### 路径 A：宿主侧自定义工具（推荐）

你的应用定义一个 `custom` 工具。智能体调用它时，会话以 `requires_action` 暂停，**你的**进程用 `pymongo` 执行查询，然后你回传一个 `user.custom_tool_result`。密钥从不进入沙箱，MongoDB 可以部署在私有 VPC 中，而智能体被限制在一组固定、可审计的查询之内。这是最轻量的路径——无需常驻服务器——也是 Anthropic 对任何含密钥数据源推荐的模式。

为了看清不被抽象遮蔽的接线方式，下面用一个针对临时 `notes` 集合的极简 `find_notes` 工具展示完整往返。**这就是核心集成：几行 `pymongo`，在宿主侧运行，藏在自定义工具之后。** 第 3 节把这一确切形态扩展到五个工具和真实工作负载。

In [3]:
notes = mongo["cookbook_mongodb_on_cma"]["notes"]
notes.delete_many({})
notes.insert_many(
    [
        {"note_id": "n1", "tag": "todo", "text": "Renew the SSL certificate before it expires."},
        {"note_id": "n2", "tag": "idea", "text": "Add a dark-mode toggle to the dashboard."},
        {"note_id": "n3", "tag": "todo", "text": "Email the quarterly report to the finance team."},
    ]
)

notes_agent = client.beta.agents.create(
    name="cookbook-mongodb-notes",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-with-mongodb-atlas"},
    model=MODEL,
    system=(
        "You answer questions about the user's notes. Use the find_notes tool to fetch notes "
        "(optionally filtered by `tag`) before answering, then give a concise final answer."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": True, "permission_policy": {"type": "always_allow"}},
        },
        {
            "type": "custom",
            "name": "find_notes",
            "description": "Return stored notes, optionally filtered by tag (e.g. 'todo', 'idea').",
            "input_schema": {
                "type": "object",
                "properties": {"tag": {"type": "string", "description": "optional tag filter"}},
                "required": [],
            },
        },
    ],
)
notes_env = client.beta.environments.create(
    name="cookbook-mongodb-notes-env", config={"type": "cloud", "networking": {"type": "limited"}}
)
notes_session = client.beta.sessions.create(
    environment_id=notes_env.id,
    agent={"type": "agent", "id": notes_agent.id, "version": notes_agent.version},
    title="MongoDB notes (host-side custom tool)",
)
print(f"seeded {notes.count_documents({})} notes; session: {notes_session.id}")

seeded 3 notes; session: sesn_016ufm58BVJ8aZeWjUt39bsq


打开事件流，提出一个问题，并驱动整个往返：每当出现 `agent.custom_tool_use`，就在**宿主侧**用 `pymongo` 执行查询，并回传一个 `user.custom_tool_result`。查询时用 `{"_id": 0}` 做投影，因为原始的 MongoDB `ObjectId` 无法序列化为 JSON。

In [4]:
def run_find_notes(args: dict) -> dict:
    """Host-side handler: pymongo runs here; _id (ObjectId) is projected out so the result is JSON-able."""
    flt = {"tag": args["tag"]} if args.get("tag") else {}
    return {"notes": list(notes.find(flt, {"_id": 0}))}


tool_use_events, responded, round_trips = {}, set(), []
question = "Using find_notes, list every note tagged 'todo', then tell me how many there are."

with client.beta.sessions.events.stream(notes_session.id) as stream:
    client.beta.sessions.events.send(
        session_id=notes_session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
    )
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded:
                        continue
                    tev = tool_use_events[event_id]
                    result = (
                        run_find_notes(tev.input)
                        if tev.name == "find_notes"
                        else {"error": "unknown"}
                    )
                    if tev.name == "find_notes":
                        round_trips.append(
                            {"tool_input": tev.input, "returned": len(result["notes"])}
                        )
                    client.beta.sessions.events.send(
                        session_id=notes_session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, notes_session.id)
print("host-side find_notes round-trips:")
for rt in round_trips:
    print(f"  find_notes({rt['tool_input']}) -> {rt['returned']} notes")

client.beta.sessions.archive(notes_session.id)
client.beta.environments.archive(notes_env.id)
client.beta.agents.archive(notes_agent.id)
notes.drop()
print("archived the demo agent / environment / session; dropped the notes collection")

host-side find_notes round-trips:
  find_notes({'tag': 'todo'}) -> 2 notes
archived the demo agent / environment / session; dropped the notes collection


### 其他选项（简述）

路径 A 是默认选择，本 cookbook 后续也使用它。当它不适用时，还有两个备选方案——跟读本指南并不需要用到它们：

- **路径 B —— 沙箱内 `pymongo`（自托管沙箱）。** 如果你在自有
  基础设施上运行沙箱，MongoDB 客户端可以位于沙箱*内部*，智能体直接用它的 `bash` 工具查询 MongoDB，`MONGO_URI` 就是一个普通的容器环境变量。只有自托管沙箱才应选这条路（在云沙箱上，连接字符串会进入会话历史）。可运行镜像：[﷐2﷑](mongodb_on_cma/self_hosted_sandbox/)。
- **路径 C —— 自托管 MongoDB MCP 服务器。** 把官方 MongoDB MCP 服务器部署在 HTTPS 之后，
  并把它注册为一个 `mcp_toolset`，这样智能体就能查询完整的能力面（`find`、`aggregate`、`$vectorSearch`），而不是一组固定查询。采用标准的 CMA MCP + 密钥库接线方式。

一句话概括权衡：**路径 A** 把查询集固定下来、密钥留在你的后端（最安全的默认选择）；**路径 B** 让沙箱直接访问；**路径 C** 给智能体最宽的查询面，代价是要运行一个服务器。拿不准时，就留在路径 A。

## 2. 检索：四种模式，一个引擎

智能体的可靠性取决于它的检索质量。下面**独立**展示 MongoDB 的四种检索模式——给出构建器，给出它的返回结果——方便你把其中某一种模式搬进自己智能体的工具里。每个构建器都只是普通的聚合工作流构建函数；第 3 节的智能体通过它的工具调用同样的这些函数。

| 模式 | 构建器 | MongoDB 阶段 |
| --- | --- | --- |
| 向量检索 | `build_vector_pipeline` | `$vectorSearch` |
| 全文检索 | `build_lexical_pipeline` | `$search` |
| 混合（倒数排名融合） | `build_rank_fusion_pipeline` | `$rankFusion`（8.0+） |
| 图遍历 | `build_graph_pipeline` | `$graphLookup` |

这些构建器在下一个单元格中内联定义——每个都返回一个聚合阶段列表，你可以直接搬进自己的集合。索引名称和被投影的字段来自 [﷐0﷑](mongodb_on_cma/config.py)；`EMBED_DIM` 和索引常量在上面环境准备单元格中导入。

首先，从 [﷐0﷑](example_data/mongodb_on_cma/seed_transactions.jsonl) 加载种子数据并创建索引。种子数据自带预先计算的嵌入向量，因此无需服务商密钥即可运行。

In [5]:
# The four retrieval builders — each returns a plain aggregation pipeline you can lift into your
# own collection. These are the functions Section 3's agent calls through its custom tools.
from mongodb_on_cma.config import (
    DECIDED_STATUSES,
    LEXICAL_PATHS,
    PROJECT_FIELDS,
    SEARCH_INDEX_NAME,
    VECTOR_INDEX_NAME,
)


def _project_stage(with_score: bool = False) -> dict:
    proj: dict[str, Any] = {f: 1 for f in PROJECT_FIELDS}
    proj["_id"] = 0
    if with_score:
        proj["score"] = {"$meta": "score"}
    return {"$project": proj}


def build_vector_pipeline(
    qvec, *, limit, candidates=None, vector_index=VECTOR_INDEX_NAME, status_in=DECIDED_STATUSES
) -> list[dict]:
    return [
        {
            "$vectorSearch": {
                "index": vector_index,
                "path": "embedding",
                "queryVector": qvec,
                "numCandidates": candidates or max(50, limit * 10),
                "limit": limit,
                "filter": {"status": {"$in": list(status_in)}},
            }
        },
        _project_stage(),
    ]


def build_lexical_pipeline(query, *, limit, search_index=SEARCH_INDEX_NAME) -> list[dict]:
    return [
        {"$search": {"index": search_index, "text": {"query": query, "path": LEXICAL_PATHS}}},
        {"$limit": limit},
        _project_stage(),
    ]


def build_rank_fusion_pipeline(
    qvec,
    query,
    *,
    k,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    status_in=DECIDED_STATUSES,
) -> list[dict]:
    # $rankFusion (MongoDB 8.0+) runs both input pipelines and fuses them by reciprocal rank —
    # one aggregation, server-side. Each input pipeline gets weight 1 (uniform). To bias toward
    # semantic vs. lexical, add "combination": {"weights": {"vector": w, "lexical": w}}.
    candidates = max(50, k * 10)
    per_branch = max(k * 4, 20)
    return [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vector": [
                            {
                                "$vectorSearch": {
                                    "index": vector_index,
                                    "path": "embedding",
                                    "queryVector": qvec,
                                    "numCandidates": candidates,
                                    "limit": per_branch,
                                    "filter": {"status": {"$in": list(status_in)}},
                                }
                            }
                        ],
                        "lexical": [
                            {
                                "$search": {
                                    "index": search_index,
                                    "text": {"query": query, "path": LEXICAL_PATHS},
                                }
                            },
                            {"$limit": per_branch},
                        ],
                    }
                },
            }
        },
        {"$limit": k},
        _project_stage(with_score=True),
    ]


def build_graph_pipeline(
    account_id: str, *, max_depth: int = 4, collection: str = "transactions"
) -> list[dict]:
    return [
        {"$match": {"sender.account_number": account_id}},
        {
            "$graphLookup": {
                "from": collection,
                "startWith": "$recipient.account_number",
                "connectFromField": "recipient.account_number",
                "connectToField": "sender.account_number",
                "as": "chain",
                "maxDepth": max_depth,
                "depthField": "depth",
            }
        },
    ]


def summarize_ring(graph_doc: dict, *, seed_account: str) -> dict:
    # Turn a $graphLookup chain into fraud-ring signals: circular flow back to the seed account,
    # layering (many small transfers), and overall network size.
    chain: list[dict] = list(graph_doc.get("chain", []))
    accounts: set[str] = set()
    small_transfers = 0
    circular_flow = False
    for edge in chain:
        sender = (edge.get("sender") or {}).get("account_number")
        recipient = (edge.get("recipient") or {}).get("account_number")
        accounts.update(a for a in (sender, recipient) if a)
        if recipient == seed_account:
            circular_flow = True
        if float(edge.get("amount", 0) or 0) < 1000:
            small_transfers += 1
    network_size = len(chain)
    layering = small_transfers >= 5
    return {
        "network_size": network_size,
        "unique_accounts": len(accounts),
        "circular_flow": circular_flow,
        "layering": layering,
        "suspicious_patterns": circular_flow or layering or network_size >= 3,
    }

In [6]:
docs = prepare_seed(load_seed(), now=datetime.now(UTC))
count = seed_collection(coll, docs)
print(f"seeded {count} transactions")

# Create the vector + Atlas Search indexes if absent, then block until both are queryable AND
# actually reflect the freshly-seeded docs (Atlas Search is eventually consistent).
ensure_indexes(coll, dim=EMBED_DIM)

check = preflight(coll)
for issue in check["issues"]:
    print("PREFLIGHT:", issue)
assert check["ok"], "Fix the preflight issues above before continuing."

# This cookbook uses the native `$rankFusion` stage for hybrid search, which needs MongoDB 8.0+.
# Every current Atlas cluster — including the free M0 tier — runs 8.0 or later, so this holds by
# default; the check just fails loudly if you point MONGO_URI at an older self-hosted server.
version = server_version(coll)
assert supports_rank_fusion(version), (
    f"MongoDB {version} predates $rankFusion (needs 8.0+). Use an Atlas cluster (any tier) or a "
    f"self-hosted MongoDB 8.0+."
)
print(f"server={version}  hybrid search=$rankFusion (native)")

seeded 20 transactions
server=8.0.27  hybrid search=$rankFusion (native)


### 模式 1：向量检索（`$vectorSearch`）

语义召回：找出 `embedding` 最接近查询向量的文档。这个示例复用某个已有文档的嵌入向量作为查询（一个待审案例在寻找已决的先例），因此不需要调用嵌入向量 API。该构建器通过 `status` 字段过滤出已决案例。

In [7]:
query_doc = coll.find_one({"transaction_id": "txn-review-struct"})  # a pending case
hits = list(coll.aggregate(build_vector_pipeline(query_doc["embedding"], limit=3)))
print("nearest decided precedents:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

nearest decided precedents:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### 模式 2：全文检索（`$search`，BM25）

对文本字段做关键词和短语匹配——这正是它能捕捉嵌入向量会模糊掉的精确名称、ID 和编码的原因。

In [8]:
hits = list(
    coll.aggregate(
        build_lexical_pipeline("cash deposit just under the reporting threshold", limit=3)
    )
)
print("full-text matches:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

full-text matches:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-review-struct: Cash deposit of 4950 USD, just under the 5000 reporting threshold, follo
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account


### 模式 3：混合（倒数排名融合）

把向量排名和全文排名融合为一个结果集。`$rankFusion`（MongoDB 8.0+）在服务端运行两个输入工作流，并按倒数排名合并——不需要客户端合并，也不需要第二次往返。

In [9]:
qvec, query = query_doc["embedding"], "structuring: cash deposit just under the threshold"
hits = list(coll.aggregate(build_rank_fusion_pipeline(qvec, query, k=3)))
print("hybrid via $rankFusion (server-side):")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

hybrid via $rankFusion (server-side):
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### 模式 4：图遍历（`$graphLookup`）

沿 `sender.account_number -> recipient.account_number` 链接追踪，以呈现一个网络（本例中是一个循环资金流团伙）。这是一个*关系*信号，有意与上面的相似度排序分开。

In [10]:
graph_doc = next(
    iter(coll.aggregate(build_graph_pipeline("ACC-RING-A", collection=coll.name))), {"chain": []}
)
ring = summarize_ring(graph_doc, seed_account="ACC-RING-A")
print(
    f"graph traversal from ACC-RING-A: network_size={ring['network_size']} "
    f"circular_flow={ring['circular_flow']} suspicious={ring['suspicious_patterns']}"
)

graph traversal from ACC-RING-A: network_size=4 circular_flow=True suspicious=True


**请为你的业务领域改造这些构建器。** 每个构建器都是通用形态：用任意模型为你的文本生成嵌入向量，并把 `build_vector_pipeline` 指向你的字段；为全文检索设置可搜索的路径；调整 `$rankFusion` 的输入权重以偏向语义或词汇一侧；把图的 `connectFromField`/`connectToField` 改成你的关系（引用关系、组织架构、供应链）。四个构建器都在上面的单元格中定义——直接搬到你自己的项目里即可。

## 3. 端到端智能体：人工介入的欺诈审核

现在所有部分协同工作。一个 Managed Agent 审核被标记的交易，把每条建议都建立在第 2 节的 MongoDB 信号之上（通过路径 A 往返以**自定义工具**形式暴露），并在高风险案例上暂停等待人工。决策和只追加的审计日志会写回同一个集群。

<p align="center"> <img src="../images/cma_with_mongodb_atlas.jpg" width="70%" alt="The agent loop runs on Anthropic; the data path runs in your notebook. MongoDB Atlas serves the retrieval signals and stores the decision and audit trail." /> </p>

**智能体循环运行在 Anthropic 侧**；**数据通路运行在你的笔记本中**。因此 `MONGO_URI` 和嵌入向量密钥永远不会进入智能体或其沙箱。等待人工的过程是持久的——会话通过 `requires_action` 在服务端空闲挂起，直到你响应——而下面的简单流式循环仅用于开发（生产环境请用 webhook 模式由持久化后端驱动）。

### AP2 授权凭证：先验证，再推理

这个智能体审核的是*由智能体发起*的支付，因此在做任何行为分析之前，它会先验证一个 **AP2（Agent Payments Protocol）** 授权凭证——这是证明用户已授权该支付的密码学签名凭证。AP2 是一个面向智能体驱动商务的[开放协议](https://github.com/google-agentic-commerce/AP2)。它的授权凭证（一个 **Checkout Mandate** 和一个 **Payment Mandate**）是防篡改的签名数字对象，构成一条可审计的链。跟读本 cookbook 不需要先学会这个协议：**调用 `verify_mandates` 工具并根据它的结论行动**（是否有效、是否满足约束、是否双重支付）。签名与验证的细节——ES256 JWT——封装在 [﷐1﷑](mongodb_on_cma/ap2_mandates.py) 中，签名后的授权凭证和回执与交易一起存放在 MongoDB 里。

下面的单元格扮演“可信侧”（生成密钥对，并把签名后的授权凭证附加到待审案例上）。私钥只保留在本地 Python 状态中——绝不进入智能体、工具结果或数据库。

In [11]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec

ts_private_key = ec.generate_private_key(ec.SECP256R1())  # Trusted Surface key, fresh each run
AGENT_PK = (
    ts_private_key.public_key()
    .public_bytes(serialization.Encoding.X962, serialization.PublicFormat.UncompressedPoint)
    .hex()
)
ts_public_key = ts_private_key.public_key()

PENDING = [
    "txn-review-clean",
    "txn-review-fraud",
    "txn-review-struct",  # seeded human-override case under AUTO_APPROVE
    "txn-review-high",
    "txn-review-ring",
]
mandates = attach_mandates(coll, PENDING, agent_pk=AGENT_PK, ts_private_key=ts_private_key)
print(f"attached AP2 mandates to {len(mandates)} transactions  agent_pk={AGENT_PK[:16]}…")

attached AP2 mandates to 5 transactions  agent_pk=04e1e32b7414d6f7…


### 自定义工具与宿主侧处理函数

每个工具都是 `type: "custom"` 工具。智能体调用其中一个时，会话暂停，**本笔记本**用 `pymongo` 运行处理函数，并把 JSON 结果发回——智能体从不接触数据库或连接字符串。处理函数的实现定义在下一个单元格：它们封装了第 2 节的构建器以及 AP2 验证（`verify_mandates`）。`HANDLERS` 只映射五个数据工具；`escalate` 有意不在其中，因为门控循环会把它路由给人工处理者而不是某个处理函数（见下一节）。

In [12]:
# The host-side tool implementations — the pymongo work behind each custom tool. These run in
# THIS process (never the sandbox), reusing the Section 2 builders. The dispatch wiring + the
# AP2/record-decision wrappers are in the next cell; build_decision_doc / build_audit_event
# (imported above) shape the persisted documents and are shared with the AP2 module.


def _jsonable(value):
    """Recursively coerce Mongo/BSON values (ObjectId, datetime, ...) into JSON-safe types."""
    if isinstance(value, dict):
        return {k: _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def _pick_fields(candidate):
    keep = ("transaction_id", "text", "amount", "sender", "recipient", "decision", "score")
    return _jsonable({k: candidate[k] for k in keep if k in candidate})


def rerank_pool_size(k: int, fanout: int, *, enable_rerank: bool) -> int:
    return k * fanout if enable_rerank else k


def merge_rerank(candidates, rerank_results, *, top_k) -> list[dict]:
    def _score(r):
        s = getattr(r, "relevance_score", None)
        return (
            (s if s is not None else r.get("relevance_score", float("-inf")))
            if isinstance(r, dict)
            else (s if s is not None else float("-inf"))
        )

    seen: set[str] = set()
    out: list[dict] = []
    for result in sorted(rerank_results, key=_score, reverse=True):
        index = getattr(result, "index", None)
        if index is None and isinstance(result, dict):
            index = result.get("index")
        score = getattr(result, "relevance_score", None)
        if score is None and isinstance(result, dict):
            score = result.get("relevance_score")
        if index is None or index < 0 or index >= len(candidates):
            continue
        candidate = candidates[index]
        cid = str(candidate.get("transaction_id", index))
        if cid in seen:
            continue
        seen.add(cid)
        out.append({**candidate, "score": score})
        if len(out) >= top_k:
            break
    return out


def tool_get_transaction(coll, transaction_id) -> dict:
    doc = coll.find_one({"transaction_id": transaction_id})
    if not doc:
        return {"error": "not_found", "transaction_id": transaction_id}
    return _jsonable({k: v for k, v in doc.items() if k not in ("embedding", "_id")})


def tool_hybrid_search_similar_frauds(
    coll,
    transaction_id,
    k,
    *,
    enable_rerank=False,
    fanout=5,
    reranker=None,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    status_in=DECIDED_STATUSES,
) -> dict:
    txn = coll.find_one({"transaction_id": transaction_id})
    if not txn:
        return {"error": "not_found", "transaction_id": transaction_id, "similar": []}
    qvec = txn["embedding"]
    query = txn.get("text", "")
    pool_k = rerank_pool_size(k, fanout, enable_rerank=enable_rerank)
    candidates = list(
        coll.aggregate(
            build_rank_fusion_pipeline(
                qvec,
                query,
                k=pool_k,
                vector_index=vector_index,
                search_index=search_index,
                status_in=status_in,
            )
        )
    )
    candidates = [c for c in candidates if str(c.get("transaction_id")) != str(transaction_id)]
    if enable_rerank and reranker is not None and candidates:
        results = reranker(query, [c.get("text", "") for c in candidates], top_k=k)
        candidates = merge_rerank(candidates, results, top_k=k)
    else:
        candidates = candidates[:k]
    return {"similar": [_pick_fields(c) for c in candidates]}


def tool_detect_fraud_ring(coll, account_id, *, max_depth=4) -> dict:
    pipeline = build_graph_pipeline(account_id, max_depth=max_depth, collection=coll.name)
    docs = list(coll.aggregate(pipeline))
    return _jsonable(summarize_ring(docs[0] if docs else {"chain": []}, seed_account=account_id))


def tool_record_decision(
    db,
    transaction_id,
    decision,
    *,
    confidence,
    risk_factors,
    reasoning,
    reviewed_by,
    escalated=False,
    recommended_decision=None,
) -> dict:
    decision_doc = build_decision_doc(
        transaction_id,
        decision,
        confidence=confidence,
        risk_factors=risk_factors,
        reasoning=reasoning,
        reviewed_by=reviewed_by,
    )
    db["transaction_decisions"].insert_one(decision_doc)
    if escalated:
        audit = build_audit_event(
            "escalated_to_human",
            transaction_id,
            decision_id=decision_doc["decision_id"],
            severity="warning",
            event_data={"human_decision": decision, "recommended_decision": recommended_decision},
        )
    else:
        audit = build_audit_event(
            "decision_stored", transaction_id, decision_id=decision_doc["decision_id"]
        )
    db["audit_events"].insert_one(audit)
    # Advance the lifecycle status to past tense (approve -> approved) so a decided case matches
    # the seed's vocabulary and DECIDED_STATUSES, making it eligible as precedent in later reviews.
    status = {"approve": "approved", "reject": "rejected"}.get(decision, decision)
    db["transactions"].update_one({"transaction_id": transaction_id}, {"$set": {"status": status}})
    return {"recorded": True, "decision_id": decision_doc["decision_id"]}

In [13]:
TOOLS = [
    {
        "type": "custom",
        "name": "verify_mandates",
        "description": "Validate the AP2 Checkout and Payment Mandate JWTs (signature, constraints, "
        "double-spend). Run this FIRST. If valid=false, constraints_satisfied=false, or "
        "double_spend_detected=true, reject immediately.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "get_transaction",
        "description": "Fetch the full transaction record under review.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "hybrid_search_similar_frauds",
        "description": "Retrieve the most similar prior (already-decided) cases as precedent, using "
        "hybrid vector + full-text search.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "k": {"type": "integer", "description": "how many precedents (default 5)"},
            },
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "detect_fraud_ring",
        "description": "Trace the account's sender->recipient chain for circular-flow / mule / layering patterns.",
        "input_schema": {
            "type": "object",
            "properties": {"account_id": {"type": "string"}},
            "required": ["account_id"],
        },
    },
    {
        "type": "custom",
        "name": "record_decision",
        "description": "Persist the final approve/reject decision with reasoning and an audit event.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "risk_factors": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "escalated": {"type": "boolean"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
            },
            "required": ["transaction_id", "decision", "reasoning"],
        },
    },
    {
        "type": "custom",
        "name": "escalate",
        "description": "Send a risky case to a human reviewer for the final decision. Use for "
        "medium-confidence, high-value, structuring, or fraud-ring cases.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "reason": {"type": "string"},
            },
            "required": ["transaction_id", "recommended_decision", "reason"],
        },
    },
]

In [14]:
# Host-side handlers — map each data tool to its handler function.
reranker = (
    (lambda q, ds, top_k: rerank(q, ds, client=ai_client, top_k=top_k))
    if (ENABLE_RERANK and ai_client)
    else None
)


def _verify_mandates(inp):
    result = tool_verify_mandates(db, inp["transaction_id"], ts_public_key)
    ok = result["valid"] and result["constraints_satisfied"] and not result["double_spend_detected"]
    print(f"\n  [verify_mandates] {inp['transaction_id']}: {'pass' if ok else 'FAIL'}")
    return result


def _hybrid(inp):
    return tool_hybrid_search_similar_frauds(
        coll,
        inp["transaction_id"],
        inp.get("k", 5),
        enable_rerank=ENABLE_RERANK,
        reranker=reranker,
    )


def _record(inp):
    result = tool_record_decision(
        db,
        inp["transaction_id"],
        inp["decision"],
        confidence=inp.get("confidence", 0),
        risk_factors=inp.get("risk_factors", []),
        reasoning=inp.get("reasoning", ""),
        reviewed_by="human" if inp.get("escalated") else "agent",
        escalated=inp.get("escalated", False),
        recommended_decision=inp.get("recommended_decision"),
    )
    if (
        inp["decision"] == "approve"
    ):  # store the AP2 receipt that later powers double-spend detection
        txn_doc = coll.find_one(
            {"transaction_id": inp["transaction_id"]},
            {"checkout_mandate_jwt": 1, "mandate_id": 1, "agent_pk": 1},
        )
        if txn_doc and txn_doc.get("mandate_id"):
            checkout_hash = hashlib.sha256(txn_doc["checkout_mandate_jwt"].encode()).hexdigest()
            store_mandate_receipt(
                db, txn_doc["mandate_id"], txn_doc["agent_pk"], checkout_hash, "approve"
            )
    return result


HANDLERS = {
    "verify_mandates": _verify_mandates,
    "get_transaction": lambda inp: tool_get_transaction(coll, inp["transaction_id"]),
    "hybrid_search_similar_frauds": _hybrid,
    "detect_fraud_ring": lambda inp: tool_detect_fraud_ring(coll, inp["account_id"]),
    "record_decision": _record,
}

### 创建智能体、环境与会话

`model`、`system` 和 `tools` 挂在**智能体**上。会话引用该智能体并配置沙箱。网络策略为 `limited`：智能体只能通过宿主侧往返访问 MongoDB。

In [15]:
SYSTEM = """You are a financial fraud reviewer. You will be given transaction IDs to review.

For EACH transaction, in order:
1. Call get_transaction to read it.
2. Call verify_mandates to validate the AP2 mandate chain (signature, constraints, double-spend).
   HARD GATE: if verify_mandates returns valid=false, constraints_satisfied=false, OR
   double_spend_detected=true, call record_decision immediately with decision="reject" and
   skip the remaining steps for that transaction.
3. Call hybrid_search_similar_frauds to retrieve similar decided precedents.
4. Call detect_fraud_ring on the sender's account_number to check for ring/mule patterns.
5. Weigh the precedents, the ring signal, the amount, and your confidence. Then make EXACTLY
   ONE terminal call for that transaction:
   - You MUST call escalate (do NOT call record_decision yourself) whenever ANY of these holds:
       * a structuring amount ($4,900-$4,999),
       * a high-value amount (>= $50,000) that you would otherwise approve,
       * detect_fraud_ring reports suspicious_patterns, or
       * your confidence is medium (~75-85).
     Give your recommended_decision and a short reason.
   - Otherwise (a clear-cut case matching none of the above), call record_decision (approve/reject).

When you escalate, you will receive the human's decision. Then call record_decision with that
decision, escalated=true, and recommended_decision set to what you had recommended.

Be concise. Move to the next transaction after recording a decision."""

agent = client.beta.agents.create(
    name="MongoDB Atlas fraud reviewer",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-with-mongodb-atlas"},
    model=MODEL,
    system=SYSTEM,
    tools=TOOLS,
)
environment = client.beta.environments.create(
    name="fraud-review-env", config={"type": "cloud", "networking": {"type": "limited"}}
)
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Fraud review",
)
print("session:", session.id)
print(f"Watch in Console: https://platform.claude.com/workspaces/default/sessions/{session.id}")

session: sesn_01LbAUC1V36CQS3Sk26eixTc
Watch in Console: https://platform.claude.com/workspaces/default/sessions/sesn_01LbAUC1V36CQS3Sk26eixTc


### 带人工门控运行审核

打开事件流，发送待审交易队列，并用 `run_gate_loop` 驱动门控循环：数据工具调用由 `HANDLERS` 自动处理，而 `escalate` 调用会暂停等待人工。当设置了 `AUTO_APPROVE`（例如在 CI 中）时，门控会确定性地解除，并且**在预置的结构化案例上覆盖智能体的判断**，让人工结论与智能体不同这一效果可见。否则，你会在单元格内被提示输入。

In [16]:
OVERRIDE_IDS = {"txn-review-struct"}


def resolve_human_decision(recommendation, *, auto_approve, override_ids, txn_id) -> str:
    """Stand in for a human reviewer. In CI (auto_approve) it concurs with the agent, except on
    the seeded override_ids where it flips the verdict so a differing human decision is visible."""
    opposite = "reject" if recommendation == "approve" else "approve"
    if auto_approve and txn_id in set(override_ids):
        return opposite
    return recommendation


def resolver(tool_input):
    txn_id = tool_input.get("transaction_id", "")
    recommended = tool_input.get("recommended_decision", "reject")
    if AUTO_APPROVE:
        decision = resolve_human_decision(
            recommended, auto_approve=True, override_ids=OVERRIDE_IDS, txn_id=txn_id
        )
        flag = "  <-- HUMAN OVERRIDE" if decision != recommended else ""
        print(f"  [human/auto] {txn_id}: agent recommended {recommended} -> human {decision}{flag}")
        return decision
    print(
        f"\n  ESCALATED {txn_id} (agent recommends {recommended}): {tool_input.get('reason', '')}"
    )
    return (
        "approve"
        if input("  Your decision [approve/reject]: ").strip().lower().startswith("a")
        else "reject"
    )


def send_result(custom_tool_use_id, result):
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.custom_tool_result",
                "custom_tool_use_id": custom_tool_use_id,
                "content": [{"type": "text", "text": json.dumps(result)}],
            }
        ],
    )


def run_gate_loop(stream, send, *, handlers, resolver) -> dict:
    """Drive the session to completion, servicing custom-tool calls as they arrive.

    This is the whole Path A round-trip for a real workload. Each `agent.custom_tool_use`
    event names a tool and its input; the session then goes idle with
    `stop_reason == "requires_action"` until we answer. We look up the tool in `handlers`
    (which run `pymongo` host-side — the credential never leaves this process), and route the
    special `escalate` tool to the human `resolver` instead. We post each result back with
    `send`, and the session resumes. Break on a terminal idle (`end_turn`) or termination.
    """
    pending: dict[str, Any] = {}
    responded: set[str] = set()
    serviced: list[tuple[str, dict]] = []
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            pending[ev.id] = ev
        elif ev.type == "session.status_idle":
            stop = getattr(ev, "stop_reason", None)
            if stop is None:
                continue
            if stop.type != "requires_action":
                break  # end_turn / retries_exhausted — the run is done
            for event_id in stop.event_ids or []:
                if event_id in responded:
                    continue
                call = pending.get(event_id)
                if call is None:
                    continue
                if call.name == "escalate":
                    result = {"human_decision": resolver(call.input)}
                else:
                    handler = handlers.get(call.name)
                    result = (
                        handler(call.input) if handler else {"error": f"unknown tool {call.name}"}
                    )
                responded.add(event_id)
                send(event_id, result)
                serviced.append((call.name, result))
        elif ev.type == "session.status_terminated":
            break
    return {"serviced": serviced, "responded": sorted(responded)}


kickoff = {
    "type": "user.message",
    "content": [
        {"type": "text", "text": "Review these flagged transactions: " + ", ".join(PENDING)}
    ],
}
run_start = datetime.now(UTC)

with client.beta.sessions.events.stream(session_id=session.id) as stream:
    client.beta.sessions.events.send(session_id=session.id, events=[kickoff])
    gate_result = run_gate_loop(stream, send_result, handlers=HANDLERS, resolver=resolver)

wait_for_idle_status(client, session.id)
print(f"\nserviced {len(gate_result['serviced'])} tool calls")


  [verify_mandates] txn-review-clean: pass

  [verify_mandates] txn-review-fraud: pass

  [verify_mandates] txn-review-struct: pass

  [verify_mandates] txn-review-high: pass

  [verify_mandates] txn-review-ring: pass

  ESCALATED txn-review-struct (agent recommends reject): Structuring indicator: cash deposit of $4,950 just under $5,000 CTR threshold, matching precedent (txn-struct-01, txn-struct-02) showing pattern of deliberate sub-threshold deposits from same account. Amount in structuring range per policy. Requires human review.


  Your decision [approve/reject]:  reject



  ESCALATED txn-review-high (agent recommends approve): High-value wire of $75,000 from Northwind Foods to Pacific Logistics for bulk shipment. Legitimate business transaction matching precedent txn-high-01 ($120,000 supplier wire, approved). Mandate valid, no fraud signals. Escalation required per policy for high-value transactions ≥ $50,000.


  Your decision [approve/reject]:  approve



  ESCALATED txn-review-ring (agent recommends reject): Money-mule ring detected. Sender ACC-RING-A (Quartz Trading) is part of a 4-account network with circular flows. Precedent txn-ring-01 (same sender/recipient, $920, rejected) shows identical pattern. Fraud ring analysis confirms suspicious_patterns, circular_flow. Requires human review for definitive ring-case determination.


  Your decision [approve/reject]:  reject



serviced 28 tool calls


### 查看决策与审计日志

从 MongoDB 读回决策和审计日志。决策集合和审计集合都是**只追加**的，因此读回范围限定在本次运行。末尾的断言让这一步成为**自检**：如果重跑无法复现预期的分流比例，就会明确报错。

人工覆盖次数反映了门控的解除方式：使用 `AUTO_APPROVE` 时它是 1（会显示对 `txn-review-struct` 预置的覆盖），而在交互式运行中，只要你在每次上报时都认同智能体的建议，它就是 0——两种结果都属正常。

In [17]:
from collections import Counter

decisions = list(db["transaction_decisions"].find({"created_at": {"$gte": run_start}}, {"_id": 0}))
audits = list(db["audit_events"].find({"timestamp": {"$gte": run_start}}, {"_id": 0}))

print("Decisions by lane:")
lanes = Counter()
for d in decisions:
    txn = coll.find_one({"transaction_id": d["transaction_id"]})
    lanes[f"{(txn['lane'] if txn else '?')} -> {d['decision']}"] += 1
for k, v in sorted(lanes.items()):
    print(f"  {k}: {v}")

overrides = [
    a
    for a in audits
    if a["event_type"] == "escalated_to_human"
    and a["event_data"].get("human_decision") != a["event_data"].get("recommended_decision")
]
print(
    f"\nescalated_to_human events: {sum(1 for a in audits if a['event_type'] == 'escalated_to_human')}"
)
print(f"human overrides (verdict != agent recommendation): {len(overrides)}")
for a in overrides:
    ed = a["event_data"]
    print(
        f"  {a['transaction_id']}: agent {ed['recommended_decision']} -> human {ed['human_decision']}"
    )

# Self-check — fail loudly if this run didn't produce the expected result.
decided_lanes = {coll.find_one({"transaction_id": d["transaction_id"]})["lane"] for d in decisions}
assert decided_lanes == {"clean_approve", "clear_reject", "high_value", "ring", "structuring"}, (
    f"expected all 5 lanes decided, saw {sorted(decided_lanes)}"
)
assert len(decisions) == len(PENDING), f"expected {len(PENDING)} decisions, got {len(decisions)}"
assert sum(1 for a in audits if a["event_type"] == "escalated_to_human") >= 1, (
    "expected >=1 escalation"
)
if AUTO_APPROVE:
    assert overrides, "AUTO_APPROVE run should show the seeded override on txn-review-struct"
print("\nself-check OK")

Decisions by lane:
  clean_approve -> approve: 1
  clear_reject -> reject: 1
  high_value -> approve: 1
  ring -> reject: 1
  structuring -> reject: 1

escalated_to_human events: 3
human overrides (verdict != agent recommendation): 0

self-check OK


## 4. MongoDB Atlas 作为记录系统与审计中枢

退后一步看看这次运行的状态存放在哪里：智能体做过的每一件事都持久化在它检索时所用的同一个集群里。这就是单一引擎在写入侧的回报——`transactions` 是运营记录（每个案例从 `pending` 推进到 `approved` 或 `rejected`），`transaction_decisions` 存放不可变的裁决，`audit_events` 是只追加的轨迹，而 `mandate_receipts` 支撑双重支付检测。一个案例的端到端过程：

In [18]:
case = "txn-review-struct"
txn = coll.find_one({"transaction_id": case}, {"_id": 0, "embedding": 0})
print(f"operational record: {case}  lane={txn['lane']}  status={txn['status']}")

decision = db["transaction_decisions"].find_one(
    {"transaction_id": case, "created_at": {"$gte": run_start}}, {"_id": 0}
)
print(
    f"decision record:    {decision['decision']}  reviewed_by={decision['reviewed_by']}  confidence={decision['confidence_score']}"
)

print("audit trail (append-only):")
for a in (
    db["audit_events"]
    .find({"transaction_id": case, "timestamp": {"$gte": run_start}}, {"_id": 0})
    .sort("timestamp", 1)
):
    extra = ""
    if a["event_type"] == "escalated_to_human":
        ed = a["event_data"]
        extra = f"  (agent recommended {ed['recommended_decision']}, human decided {ed['human_decision']})"
    print(f"  {a['timestamp']:%H:%M:%S}  {a['event_type']}  severity={a['severity']}{extra}")

txn_mandate = coll.find_one({"transaction_id": case}, {"mandate_id": 1, "agent_pk": 1})
if txn_mandate and txn_mandate.get("mandate_id"):
    receipt = db["mandate_receipts"].find_one({"mandate_id": txn_mandate["mandate_id"]}, {"_id": 0})
    if receipt:
        print(
            f"mandate receipt:    mandate_id={receipt['mandate_id'][:12]}…  decision={receipt['decision']}"
        )

operational record: txn-review-struct  lane=structuring  status=rejected
decision record:    reject  reviewed_by=human  confidence=0.9
audit trail (append-only):
  13:49:22  escalated_to_human  severity=warning  (agent recommended reject, human decided reject)


**生产环境用 webhook（仅指路，此处不运行）：** 上面的流式循环很适合开发，但无法在重启后存活。生产环境请为 `session.status_idled` 注册一个 webhook：你的处理函数把案例排入人工审核队列，之后再 POST 那个 `user.custom_tool_result`。**数据工具的处理函数完全相同**，改变的只是触发方式。参见 [﷐2﷑](CMA_operate_in_production.ipynb)。

In [19]:
client.beta.sessions.archive(session.id)
client.beta.environments.archive(environment.id)
# Agents are reusable across runs; archiving is optional and permanent.
mongo.close()
print("archived session + environment; closed MongoDB connection")

archived session + environment; closed MongoDB connection


## 5. 回顾

你把 MongoDB Atlas 接入了 Claude Managed Agent，并在一次构建中走完了从连接到记录系统的全过程：

- **连接** —— 三条不泄露凭证的数据通路（宿主侧自定义工具、自托管沙箱、
  自托管 MCP），MongoDB 密钥始终在边界的你这一侧。
- **检索** —— 向量、全文、混合（倒数排名融合）和图遍历，都是可提取的聚合流水线
  构建器，已在第 2 节内联定义。
- **门控** —— 高风险决策由 CMA 原生的 `requires_action` 人工暂停来把关。
- **持久化** —— 裁决、只追加的审计轨迹以及 AP2 回执，都存放在智能体检索所用的
  同一个集群中。

整个集成的接触面很小：自定义工具背后只有几行 `pymongo`。换掉集合和工具，同样的形态就能把任意智能体扎根于 MongoDB。检索构建器、工具处理函数和门控循环全都就在这本笔记本里；[﷐1﷑](mongodb_on_cma/) 包只存放环境准备的样板代码。